In this part , the task is to use pretrained model, streamlit and put it in docker

In [11]:
import torch
import torch.nn as nn
from torchvision import models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

In [12]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


train_dir = r"C:\Users\walaa\Desktop\Side Projects\AI\Computer Vision\Cellula\Teeth Classification\Teeth_Dataset\Training"
val_dir   = r"C:\Users\walaa\Desktop\Side Projects\AI\Computer Vision\Cellula\Teeth Classification\Teeth_Dataset\Validation"
test_dir  = r"C:\Users\walaa\Desktop\Side Projects\AI\Computer Vision\Cellula\Teeth Classification\Teeth_Dataset\Testing"


train_dataset = datasets.ImageFolder(train_dir, transform=transform)
val_dataset   = datasets.ImageFolder(val_dir, transform=transform)
test_dataset  = datasets.ImageFolder(test_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

num_classes = len(train_dataset.classes)

print("Classes:", train_dataset.classes)
print("Number of classes:", num_classes)


Classes: ['CaS', 'CoS', 'Gum', 'MC', 'OC', 'OLP', 'OT']
Number of classes: 7


In [13]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

In [14]:
num_classes = 7
num_features = model.fc.in_features

model.fc = nn.Linear(num_features, num_classes)

for param in model.parameters():
    param.requires_grad = False
for param in model.fc.parameters():
    param.requires_grad = True

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [16]:
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)

In [21]:
num_epochs = 10
criterion = nn.CrossEntropyLoss()

for epoch in range(num_epochs):
    model.train()
    running_loss = 0
    correct = 0
    total = 0


    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=True)
    
    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        loop.set_postfix(loss=running_loss/total, accuracy=correct/total)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    print(f"Epoch {epoch+1} finished: Loss={epoch_loss:.4f}, Accuracy={epoch_acc:.4f}\n")


Epoch 1/10: 100%|██████████| 97/97 [01:41<00:00,  1.05s/it, accuracy=0.844, loss=0.466]


Epoch 1 finished: Loss=0.4661, Accuracy=0.8442



Epoch 2/10: 100%|██████████| 97/97 [01:41<00:00,  1.04s/it, accuracy=0.854, loss=0.443]


Epoch 2 finished: Loss=0.4431, Accuracy=0.8539



Epoch 3/10: 100%|██████████| 97/97 [01:41<00:00,  1.04s/it, accuracy=0.852, loss=0.444]


...

Epoch 8 finished: Loss=0.4110, Accuracy=0.8672



Epoch 9/10: 100%|██████████| 97/97 [01:36<00:00,  1.00it/s, accuracy=0.861, loss=0.42] 


Epoch 9 finished: Loss=0.4198, Accuracy=0.8610



Epoch 10/10: 100%|██████████| 97/97 [01:48<00:00,  1.12s/it, accuracy=0.868, loss=0.397]

Epoch 10 finished: Loss=0.3971, Accuracy=0.8685



In [22]:
model.eval()
correct = 0
total = 0

for images, labels in tqdm(val_loader, desc="Validating"):
    images, labels = images.to(device), labels.to(device)
    outputs = model(images)
    _, predicted = torch.max(outputs, 1)
    correct += (predicted == labels).sum().item()
    total += labels.size(0)

val_acc = correct / total
print(f"Validation Accuracy: {val_acc:.4f}")


Validating: 100%|██████████| 33/33 [00:29<00:00,  1.14it/s]

Validation Accuracy: 0.8385


In [23]:
torch.save(model.state_dict(), "resnet_transferL2.pth")
